# 01 Candidate Generation — FAISS embedding blocking

Цель ноутбука — первичный blocking для дедупликации SKU: для каждого товара найти top-k ближайших соседей по dense embeddings через FAISS. На выходе получается список пар-кандидатов, а не финальный ответ дубль/не дубль.

## План

1. Загрузить срез категории `Соусы` из `mpstats_products`.
2. Привести строки к research-контракту `marketplace + Артикул`.
3. Построить embedding-текст из `brand + title`.
4. Посчитать dense embeddings через `SentenceTransformer`.
5. Построить FAISS `IndexFlatIP` по L2-normalized vectors и взять top-k соседей.
6. Сохранить `research/dedup/data/candidates_sauces.csv` для ручной разметки и дальнейшего matching comparison.

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys
import time

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    FAISS_CANDIDATE_OUTPUT_COLUMNS,
    CandidateGenerationConfig,
    FaissCandidateGenerationConfig,
    generate_faiss_candidate_pairs,
    prepare_product_records,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 140)

In [ ]:
TARGET_CATEGORY = "Соусы"
CATEGORY_ALIASES = ["Соусы", "Соус"]
PRODUCTS_TABLE = "mpstats_products"
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
CANDIDATES_PATH = Path(os.environ.get("DEDUP_CANDIDATES_PATH", DATA_DIR / "candidates_sauces.csv")).expanduser()

EMBEDDING_MODEL = os.environ.get("DEDUP_EMBEDDING_MODEL", "intfloat/multilingual-e5-small")
EMBEDDING_BATCH_SIZE = int(os.environ.get("DEDUP_EMBEDDING_BATCH_SIZE", "64"))
FAISS_TOP_K = int(os.environ.get("DEDUP_FAISS_TOP_K", "20"))
MAX_CANDIDATES = int(os.environ.get("DEDUP_FAISS_MAX_CANDIDATES", "60000"))
_record_limit_raw = os.environ.get("DEDUP_FAISS_RECORD_LIMIT", "").strip()
RECORD_LIMIT = int(_record_limit_raw) if _record_limit_raw else None
_min_similarity_raw = os.environ.get("DEDUP_FAISS_MIN_SIMILARITY", "").strip()
MIN_SIMILARITY = float(_min_similarity_raw) if _min_similarity_raw else None

feature_config = CandidateGenerationConfig(max_candidates=None)
faiss_config = FaissCandidateGenerationConfig(
    top_k=FAISS_TOP_K,
    max_candidates=MAX_CANDIDATES,
    min_similarity=MIN_SIMILARITY,
    candidate_features=feature_config,
)

def resolve_duckdb_path(project_root: Path) -> Path:
    env_path = os.environ.get("MPSTATS_DUCKDB_PATH")
    candidates = [Path(env_path).expanduser() if env_path else None, project_root / "mpstats.duckdb"]
    for candidate in candidates:
        if candidate is not None and candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "DuckDB-куб не найден. Задайте MPSTATS_DUCKDB_PATH=/absolute/path/to/mpstats.duckdb "
        "или положите mpstats.duckdb в корень проекта."
    )

missing = [
    package
    for package, module_name in {"faiss-cpu": "faiss", "sentence-transformers": "sentence_transformers"}.items()
    if importlib.util.find_spec(module_name) is None
]
if missing:
    raise ImportError(
        "Для FAISS embedding blocking нужны research-зависимости: "
        + ", ".join(missing)
        + ". Установите: python3 -m pip install -r requirements-research.txt"
    )

# Import FAISS before sentence-transformers/torch. On local macOS/Python 3.13,
# importing FAISS after torch can crash the process in native code.
import faiss

DB_PATH = resolve_duckdb_path(PROJECT_ROOT)
print(f"DuckDB cube: {DB_PATH}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"FAISS top_k: {FAISS_TOP_K}; max candidates: {MAX_CANDIDATES}; min similarity: {MIN_SIMILARITY}")
print(f"Record limit: {RECORD_LIMIT}")
print(f"Candidates output: {CANDIDATES_PATH}")

In [ ]:
with duckdb.connect(str(DB_PATH), read_only=True) as con:
    available_categories = con.execute(
        f'SELECT DISTINCT "Категория" FROM {PRODUCTS_TABLE} ORDER BY 1'
    ).fetchdf()["Категория"].dropna().tolist()
    real_category = next((category for category in CATEGORY_ALIASES if category in available_categories), None)
    if real_category is None:
        raise ValueError(f"Категория {TARGET_CATEGORY!r} не найдена. Доступно: {available_categories[:20]}")
    products_df = con.execute(
        f'SELECT * FROM {PRODUCTS_TABLE} WHERE "Категория" = ?',
        [real_category],
    ).fetchdf()

print(f"Resolved category: {real_category}")
print(f"Loaded rows: {len(products_df):,}")
display(products_df.head(3))

## Product records

Один record — это `marketplace + Артикул`. Повторы по месяцам внутри одного marketplace агрегируются, чтобы FAISS искал соседей между товарами, а не между месячными строками одного товара.

In [ ]:
product_records = prepare_product_records(products_df, feature_config)
if RECORD_LIMIT is not None:
    product_records = product_records.head(RECORD_LIMIT).copy()
    print(f"Using first {len(product_records):,} product records because DEDUP_FAISS_RECORD_LIMIT is set")
articles_by_marketplaces = product_records.groupby("sku")["marketplace"].nunique(dropna=True)
summary = pd.DataFrame(
    [
        {
            "raw_rows": len(products_df),
            "product_records": len(product_records),
            "unique_articles": product_records["sku"].nunique(dropna=True),
            "marketplaces": product_records["marketplace"].nunique(dropna=True),
            "articles_seen_in_multiple_marketplaces": int((articles_by_marketplaces > 1).sum()),
            "unique_titles": product_records["title_norm"].nunique(dropna=True),
            "brand_fill_share": product_records["brand_norm"].ne("").mean(),
            "multipack_gt_1_share": (pd.to_numeric(product_records["multipack_count"], errors="coerce") > 1).mean(),
        }
    ]
)
display(summary)
display(product_records.head(5))

## Embeddings

Embedding-текст собирается из бренда и title. Для E5-моделей добавляется `passage:` prefix, потому что это стандартный формат их retrieval-входа.

In [ ]:
from sentence_transformers import SentenceTransformer

def build_embedding_text(row: pd.Series) -> str:
    parts = [str(row.get("brand") or "").strip(), str(row.get("title") or "").strip()]
    text = " ".join(part for part in parts if part)
    if "e5" in EMBEDDING_MODEL.lower():
        return f"passage: {text}"
    return text

embedding_texts = product_records.apply(build_embedding_text, axis=1).tolist()
display(pd.DataFrame({"embedding_text": embedding_texts[:5]}))

model = SentenceTransformer(EMBEDDING_MODEL)
started_at = time.perf_counter()
embeddings = model.encode(
    embedding_texts,
    batch_size=EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
elapsed_sec = time.perf_counter() - started_at
print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding time: {elapsed_sec:.2f} sec")

## FAISS top-k blocking

Здесь и происходит первичный поиск кандидатов: FAISS ищет top-k ближайших embedding-соседей для каждого product record. `baseline_similarity_score` оставлен только как совместимое имя для следующих ноутбуков; его значение равно `embedding_similarity_score`.

In [ ]:
started_at = time.perf_counter()
candidates = generate_faiss_candidate_pairs(product_records, embeddings, faiss_config, faiss_module=faiss)
elapsed_sec = time.perf_counter() - started_at

print(f"Candidate pairs: {len(candidates):,}")
print(f"FAISS generation time: {elapsed_sec:.2f} sec")
display(candidates.head(10))

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
candidates_to_save = candidates[FAISS_CANDIDATE_OUTPUT_COLUMNS].copy()
candidates_to_save.to_csv(CANDIDATES_PATH, index=False)

print(f"Saved candidates: {CANDIDATES_PATH}")
print(f"Rows saved: {len(candidates_to_save):,}")
display(candidates_to_save.head(5))

pair_scope_stats = (
    candidates_to_save.assign(
        pair_scope=candidates_to_save["is_cross_marketplace_pair"].map(
            {True: "cross_marketplace", False: "same_marketplace"}
        )
    )["pair_scope"]
    .value_counts()
    .rename_axis("pair_scope")
    .reset_index(name="pairs")
)
display(pair_scope_stats)

## Embedding score distribution

In [ ]:
score_summary = candidates_to_save["embedding_similarity_score"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).to_frame("embedding_similarity_score")
display(score_summary)

ax = candidates_to_save["embedding_similarity_score"].hist(bins=40, figsize=(10, 4), color="#4C78A8")
ax.set_title("Distribution of FAISS embedding cosine score")
ax.set_xlabel("embedding_similarity_score")
ax.set_ylabel("candidate pairs")
plt.tight_layout()
plt.show()

## Sanity checks

Hard-negative и pack-variant флаги ниже не создают пары. Они только помечают уже найденные FAISS пары, чтобы `02_labeling_dataset.ipynb` мог выбрать полезную ручную разметку.

In [ ]:
flag_stats = pd.DataFrame(
    [
        {"flag": "is_cross_marketplace_pair", "pairs": int(candidates_to_save["is_cross_marketplace_pair"].fillna(False).sum())},
        {"flag": "is_hard_negative_candidate", "pairs": int(candidates_to_save["is_hard_negative_candidate"].fillna(False).sum())},
        {"flag": "is_pack_variant_candidate", "pairs": int(candidates_to_save["is_pack_variant_candidate"].fillna(False).sum())},
    ]
)
display(flag_stats)

def show_examples(frame: pd.DataFrame, label: str, n: int = 5) -> None:
    if frame.empty:
        print(f"{label}: нет примеров")
        return
    print(label)
    display(
        frame[[
            "raw_record_id_a",
            "raw_record_id_b",
            "marketplace_a",
            "marketplace_b",
            "sku_a",
            "sku_b",
            "title_a",
            "title_b",
            "brand_a",
            "brand_b",
            "embedding_similarity_score",
            "candidate_rank",
            "is_cross_marketplace_pair",
            "is_hard_negative_candidate",
            "is_pack_variant_candidate",
        ]].head(n)
    )

show_examples(candidates_to_save.head(20), "Top FAISS candidates")
show_examples(candidates_to_save[candidates_to_save["is_cross_marketplace_pair"]], "Cross-marketplace candidates")
show_examples(candidates_to_save[candidates_to_save["is_hard_negative_candidate"]], "Hard-negative candidates")
show_examples(candidates_to_save[candidates_to_save["is_pack_variant_candidate"]], "Pack-variant candidates")

## Итог

- `candidates_sauces.csv` теперь строится через dense embeddings + FAISS top-k.
- `baseline_similarity_score` в этом файле — это совместимое имя для embedding cosine score.
- Следующий шаг: запустить `02_labeling_dataset.ipynb`, разметить `label`, затем сравнивать методы pairwise matching в `03_matching_comparison.ipynb`.